# ModelForge Qwen LoRA evidence run on Google Colab

This notebook fine-tunes the pinned Qwen model and runs the minimum serving evidence needed for the implementation claim, entirely inside a hosted Colab VM. It does not mount Google Drive, upload files from your computer, or copy the base-model weights into the evidence repository. Qwen's weights live only in Colab's temporary cache and disappear when you delete the runtime.

**This check must happen before you press Connect:** inspect the Colab compute-unit balance and obtain explicit approval to deduct this run from that existing balance. Then choose **Runtime → Change runtime type**, select the **2026.07** past runtime, and request a standard-memory **T4**. Do not purchase more units, upgrade the plan, or select Premium GPU, High RAM, a GCP marketplace runtime, or an Enterprise runtime. Notebook code cannot inspect billing because code runs only after a runtime is allocated; the checkbox below is an attestation, not a technical billing control. If any condition is unclear or Colab requests a purchase, do not connect. Create a private Hugging Face **dataset** repository for artifacts, then add a short-lived fine-grained secret named `HF_TOKEN` with write access only to that repository. Never paste the token into a cell.

This notebook is prepared but has not yet produced a model or metrics. A successful run preserves training, one-case preflight, response-mask, FastAPI/Pydantic, routing, environment, and checksum evidence; it deliberately stops before the locked test split. It does not measure or claim model-quality improvement or calibrated routing.

## 1. Set the three run-specific values

Use the exact 40-character Git commit containing the reviewed workflow. Choose a fresh lowercase run ID; the notebook refuses to overwrite a prior remote run.

In [ ]:
import json
import os
import platform
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rubiatExe/ModelForge.git"
SOURCE_COMMIT = "REPLACE_WITH_40_CHARACTER_COMMIT_SHA"  # @param {type:"string"}
ARTIFACT_REPO = "YOUR_HF_USERNAME/modelforge-colab-artifacts"  # @param {type:"string"}
RUN_ID = "qwen-lora-r16-20260906-01"  # @param {type:"string"}
INCLUDED_UNITS_APPROVED = False  # @param {type:"boolean"}
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"

PROJECT_ROOT = Path("/content/ModelForge")
VENV_ROOT = Path("/content/modelforge-venv")
OUTPUT_ROOT = Path("/content/modelforge-output")
TRAINING_OUTPUT_ROOT = OUTPUT_ROOT / "training-runs"
BUNDLE_PARENT = Path("/content/modelforge-bundles")
DERIVED_CONFIG = Path("/content/modelforge-configs") / f"{RUN_ID}.yaml"
REMOTE_RUN_PATH = f"runs/{SOURCE_COMMIT}/{RUN_ID}"

if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
    raise ValueError("Replace SOURCE_COMMIT with the reviewed 40-character commit SHA")
if ARTIFACT_REPO.startswith("YOUR_") or ARTIFACT_REPO.count("/") != 1:
    raise ValueError("Replace ARTIFACT_REPO with namespace/private-dataset-repository")
if re.fullmatch(r"[a-z0-9][a-z0-9._-]{2,99}", RUN_ID) is None:
    raise ValueError("RUN_ID must be a fresh lowercase, path-safe identifier")
if not INCLUDED_UNITS_APPROVED:
    raise RuntimeError(
        "Stop: confirm explicit approval to use only the existing included-unit balance, "
        "with no purchase, plan upgrade, Premium GPU, High RAM, GCP, or Enterprise "
        "runtime. Code cannot inspect billing; if unclear, do not continue."
    )
if sys.version_info[:3] != (3, 12, 13):
    raise RuntimeError("Select Colab runtime version 2026.07 (Python 3.12.13)")
if sys.platform != "linux" or platform.machine() != "x86_64":
    raise RuntimeError("This lock is only for Colab Linux x86_64")
if Path("/content/drive").is_mount():
    raise RuntimeError("Google Drive is mounted; use a fresh runtime with Drive disconnected")

os.environ["HF_HOME"] = "/content/hf-model-cache"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["DO_NOT_TRACK"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gpu_row = (
    subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        text=True,
    )
    .strip()
    .splitlines()[0]
)
gpu_name, gpu_memory = (part.strip() for part in gpu_row.rsplit(",", 1))
if "T4" not in gpu_name:
    raise RuntimeError(f"Expected the approved standard-memory T4; found {gpu_name}")
if int(gpu_memory) < 14000:
    raise RuntimeError(f"At least 14 GB of GPU memory is required; found {gpu_memory} MiB")
print({"gpu": gpu_name, "memory_mib": int(gpu_memory), "remote_path": REMOTE_RUN_PATH})

## 2. Fetch and verify only the pinned source commit

This creates a fresh checkout and fails if the runtime has old state. No local computer directory is uploaded.

In [ ]:
for fresh_path in (PROJECT_ROOT, VENV_ROOT, OUTPUT_ROOT, BUNDLE_PARENT):
    if fresh_path.exists() or fresh_path.is_symlink():
        raise RuntimeError(
            f"{fresh_path} already exists; disconnect and delete the runtime before retrying"
        )
PROJECT_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(["git", "init", str(PROJECT_ROOT)], check=True)
subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "remote", "add", "origin", REPO_URL],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "fetch", "--depth=1", "origin", SOURCE_COMMIT],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", "FETCH_HEAD"],
    check=True,
)
actual_origin = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"], text=True
).strip()
actual_head = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
).strip()
dirty = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain=v1", "--untracked-files=all"],
    text=True,
).strip()
if actual_origin != REPO_URL or actual_head != SOURCE_COMMIT or dirty:
    raise RuntimeError("Pinned Git checkout verification failed")
print("Verified clean source commit:", actual_head)

## 3. Install the exact reviewed Linux/CUDA environment

The lock contains exact versions and SHA-256 hashes. Installation happens before any Hugging Face credential is read.

In [ ]:
LOCK_PATH = PROJECT_ROOT / "requirements/colab-linux-py312-cu128.lock"
CONFIG_TEMPLATE = PROJECT_ROOT / "experiments/configs/qwen_lora_r16_colab_v1.yaml"
HELPER_PATH = PROJECT_ROOT / "scripts/colab_workflow.py"
NOTEBOOK_SOURCE = PROJECT_ROOT / "notebooks/modelforge_qwen_lora_colab.ipynb"
for required in (LOCK_PATH, CONFIG_TEMPLATE, HELPER_PATH, NOTEBOOK_SOURCE):
    if not required.is_file() or required.is_symlink():
        raise RuntimeError(f"Missing committed workflow file: {required}")

subprocess.run(
    [sys.executable, "-m", "venv", "--without-pip", str(VENV_ROOT)], check=True
)
VENV_PYTHON = VENV_ROOT / "bin/python"
VENV_PIP = [sys.executable, "-m", "pip", "--python", str(VENV_PYTHON)]
subprocess.run(
    [
        *VENV_PIP,
        "install",
        "--disable-pip-version-check",
        "--no-input",
        "--require-hashes",
        "--only-binary=:all:",
        "--index-url",
        "https://pypi.org/simple",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu128",
        "--requirement",
        str(LOCK_PATH),
    ],
    check=True,
)
subprocess.run([*VENV_PIP, "check"], check=True)

run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run(
    [str(VENV_PYTHON), "-m", "ruff", "check", "."],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
subprocess.run(
    [str(VENV_PYTHON), "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
cuda_probe = subprocess.check_output(
    [
        str(VENV_PYTHON),
        "-c",
        "import torch; assert torch.cuda.is_available(); "
        "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))",
    ],
    text=True,
).strip()
print("Verified locked environment:", cuda_probe)

## 4. Re-verify source, derive a unique config, and record the environment

In [ ]:
venv_site = json.loads(
    subprocess.check_output(
        [str(VENV_PYTHON), "-c", "import json, site; print(json.dumps(site.getsitepackages()))"],
        text=True,
    )
)[0]
for import_path in (venv_site, str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)):
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

from scripts.colab_workflow import (  # noqa: E402
    derive_cloud_config,
    prepare_evidence_bundle,
    sha256_file,
    validate_inputs,
    verify_downloaded_bundle,
    verify_git_checkout,
    verify_response_only_mask_audit,
    verify_training_manifest,
)

validated_inputs = validate_inputs(
    source_commit=SOURCE_COMMIT, artifact_repo_id=ARTIFACT_REPO, run_id=RUN_ID
)
checkout = verify_git_checkout(
    PROJECT_ROOT,
    expected_origin=REPO_URL,
    expected_head=SOURCE_COMMIT,
    required_files=(LOCK_PATH, CONFIG_TEMPLATE, HELPER_PATH, NOTEBOOK_SOURCE),
)
derive_cloud_config(
    CONFIG_TEMPLATE,
    DERIVED_CONFIG,
    checkout=checkout,
    output_root=TRAINING_OUTPUT_ROOT,
    run_id=RUN_ID,
)

environment_root = OUTPUT_ROOT / "environment"
environment_root.mkdir(parents=True, exist_ok=False)
pip_check = subprocess.run(
    [*VENV_PIP, "check"],
    check=True,
    capture_output=True,
    text=True,
)
(environment_root / "pip-check.txt").write_text(pip_check.stdout, encoding="utf-8")
with (environment_root / "pip-freeze.txt").open("w", encoding="utf-8") as handle:
    subprocess.run(
        [*VENV_PIP, "freeze", "--all"],
        stdout=handle,
        text=True,
        check=True,
    )
with (environment_root / "nvidia-smi.txt").open("w", encoding="utf-8") as handle:
    subprocess.run(["nvidia-smi", "-q"], stdout=handle, text=True, check=True)
runtime_probe = subprocess.check_output(
    [
        str(VENV_PYTHON),
        "-c",
        "import json, platform, sys, torch; "
        "print(json.dumps({'python': sys.version, 'platform': platform.platform(), "
        "'torch': torch.__version__, 'cuda_runtime': torch.version.cuda, "
        "'cuda_available': torch.cuda.is_available(), "
        "'gpu': torch.cuda.get_device_name(0)}))",
    ],
    text=True,
)
runtime_record = json.loads(runtime_probe)
runtime_record.update(
    {
        "source_repository": REPO_URL,
        "source_commit": SOURCE_COMMIT,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "dependency_lock_sha256": sha256_file(LOCK_PATH),
        "colab_runtime_version": "2026.07",
        "installer_pip": subprocess.check_output(
            [sys.executable, "-m", "pip", "--version"], text=True
        ).strip(),
        "pytorch_cuda_alloc_conf": run_env["PYTORCH_CUDA_ALLOC_CONF"],
        "included_compute_units_user_approved": INCLUDED_UNITS_APPROVED,
        "new_purchase_authorized": False,
        "billing_inspected_by_notebook": False,
    }
)
(environment_root / "runtime.json").write_text(
    json.dumps(runtime_record, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
(environment_root / "source-commit.txt").write_text(SOURCE_COMMIT + "\n", encoding="utf-8")
(environment_root / "dependency-lock-sha256.txt").write_text(
    sha256_file(LOCK_PATH) + "\n", encoding="utf-8"
)
environment_files = {path.name: path for path in sorted(environment_root.iterdir())}
print(json.dumps(runtime_record, indent=2, sort_keys=True))

## 5. Run a one-case base-model preflight

This is the first step that downloads Qwen. The download goes to `/content/hf-model-cache`, outside the source and evidence bundle. The preflight catches model-loading or CUDA failures before training.

In [ ]:
preflight_result = Path("/content/qwen-base-preflight.json")
subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "modelforge.experiments.run_evaluation",
        "--backend",
        "hf-base",
        "--experiment-id",
        f"{RUN_ID}-base-preflight",
        "--split",
        "validation",
        "--limit",
        "1",
        "--dataset-root",
        str(PROJECT_ROOT / "data/iam_triage_v1"),
        "--output",
        str(preflight_result),
        "--model-name-or-path",
        MODEL_ID,
        "--revision",
        MODEL_REVISION,
        "--serving-id",
        "small-iam-triage-v1",
        "--device",
        "cuda",
        "--precision",
        "auto",
        "--max-new-tokens",
        "256",
    ],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
preflight = json.loads(preflight_result.read_text(encoding="utf-8"))
if preflight["operations"]["attempted_cases"] != 1:
    raise RuntimeError("Preflight did not attempt exactly one validation case")
if preflight["operations"]["operational_errors"] != 0:
    raise RuntimeError("Preflight recorded a model-loading or inference error")
run_env["HF_HUB_OFFLINE"] = "1"
run_env["TRANSFORMERS_OFFLINE"] = "1"
print("Base-model CUDA preflight passed; subsequent model loads are cache-only")

## 6. Audit response-only masking and routing control flow

The first command uses the cached, commit-pinned Qwen fast tokenizer on every manifest-verified training row and writes no text or token IDs. It fails if any prompt label leaks, any row is skipped, or any sequence truncates. The second command exercises the real router with deterministic fixtures; its artifact is explicitly control-flow evidence, not model-quality or calibration evidence.

In [ ]:
audit_root = OUTPUT_ROOT / "audits"
audit_root.mkdir(parents=True, exist_ok=False)
response_mask_audit = audit_root / "response-only-mask-audit.json"
routing_smoke = audit_root / "confidence-routing-smoke.json"
subprocess.run(
    [
        str(VENV_PYTHON),
        str(HELPER_PATH),
        "response-mask-audit",
        "--config",
        str(DERIVED_CONFIG),
        "--output",
        str(response_mask_audit),
        "--project-root",
        str(PROJECT_ROOT),
        "--source-commit",
        SOURCE_COMMIT,
        "--run-id",
        RUN_ID,
    ],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
subprocess.run(
    [
        str(VENV_PYTHON),
        str(HELPER_PATH),
        "confidence-routing-smoke",
        "--output",
        str(routing_smoke),
        "--project-root",
        str(PROJECT_ROOT),
        "--source-commit",
        SOURCE_COMMIT,
        "--run-id",
        RUN_ID,
    ],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
for artifact_path in (response_mask_audit, routing_smoke):
    artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
    if not (artifact.get("passed") or artifact.get("all_passed")):
        raise RuntimeError(f"Evidence stage did not pass: {artifact_path.name}")
print("Verified full response-mask audit and uncalibrated routing control-flow smoke")

## 7. Fine-tune the response-only LoRA adapter and verify its manifest

In [ ]:
subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "modelforge.training.train_lora",
        "--config",
        str(DERIVED_CONFIG),
        "--project-root",
        str(PROJECT_ROOT),
    ],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
run_root = TRAINING_OUTPUT_ROOT / RUN_ID
manifest_path = run_root / "manifest.json"
training_summary = verify_training_manifest(
    manifest_path, source_commit=SOURCE_COMMIT, run_id=RUN_ID
)
mask_audit_summary = verify_response_only_mask_audit(
    response_mask_audit,
    training_manifest_path=manifest_path,
    source_commit=SOURCE_COMMIT,
    run_id=RUN_ID,
)
print("Verified completed training:")
print(
    json.dumps(
        {key: str(value) for key, value in training_summary.items() if key != "artifact_hashes"},
        indent=2,
    )
)
print(json.dumps(mask_audit_summary, indent=2, sort_keys=True, default=str))

## 8. Upload an allowlisted training bundle to the private repository

The secret is read only inside the upload function. It is never placed in an environment variable, printed, or passed to training/model subprocesses.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

repository_head = None
remote_initialized = False
upload_receipts = []


def persist_evidence(bundle_root: Path, *, phase: str, first_upload: bool) -> dict[str, str]:
    global remote_initialized, repository_head
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Add a short-lived, repository-scoped HF_TOKEN in Colab Secrets")
    api = HfApi(token=token)
    try:
        info = api.repo_info(repo_id=ARTIFACT_REPO, repo_type="dataset")
        if not bool(getattr(info, "private", False)):
            raise RuntimeError(
                "Refusing to upload: the Hugging Face dataset repository is not private"
            )
        current_head = getattr(info, "sha", None)
        if repository_head is not None and current_head != repository_head:
            raise RuntimeError("Artifact repository changed concurrently; refusing a stale upload")
        remote_files = api.list_repo_files(
            repo_id=ARTIFACT_REPO, repo_type="dataset", revision=current_head
        )
        prefix = REMOTE_RUN_PATH + "/"
        existing = {name[len(prefix) :] for name in remote_files if name.startswith(prefix)}
        if first_upload and existing:
            raise RuntimeError("Remote run prefix already exists; choose a fresh RUN_ID")
        if first_upload != (not remote_initialized):
            raise RuntimeError("Upload phase ordering is invalid")
        local_files = sorted(
            path.relative_to(bundle_root).as_posix()
            for path in bundle_root.rglob("*")
            if path.is_file()
        )
        commit = api.upload_folder(
            repo_id=ARTIFACT_REPO,
            repo_type="dataset",
            folder_path=str(bundle_root),
            path_in_repo=REMOTE_RUN_PATH,
            allow_patterns=local_files,
            parent_commit=current_head,
            commit_message=f"{RUN_ID}: preserve {phase}",
        )
        final_info = api.repo_info(repo_id=ARTIFACT_REPO, repo_type="dataset")
        final_head = getattr(final_info, "sha", None)
        if not isinstance(final_head, str) or re.fullmatch(r"[0-9a-f]{40}", final_head) is None:
            raise RuntimeError("Hugging Face did not return a commit-pinned repository state")
        final_files = api.list_repo_files(
            repo_id=ARTIFACT_REPO, repo_type="dataset", revision=final_head
        )
        uploaded = {name[len(prefix) :] for name in final_files if name.startswith(prefix)}
        if uploaded != set(local_files):
            raise RuntimeError("Remote run prefix differs from the reviewed local bundle")
        repository_head = final_head
        remote_initialized = True
        receipt = {
            "phase": phase,
            "upload_commit": str(getattr(commit, "oid", "")),
            "verified_repo_head": final_head,
            "url": f"https://huggingface.co/datasets/{ARTIFACT_REPO}/tree/{final_head}/{REMOTE_RUN_PATH}",
        }
        print(json.dumps(receipt, indent=2, sort_keys=True))
        return receipt
    finally:
        token = ""
        api = None


training_bundle = prepare_evidence_bundle(
    manifest_path,
    BUNDLE_PARENT / "training",
    source_commit=SOURCE_COMMIT,
    run_id=RUN_ID,
    environment_files=environment_files,
    evaluation_files={"base-preflight.json": preflight_result},
    audit_files={
        "response-only-mask-audit.json": response_mask_audit,
        "confidence-routing-smoke.json": routing_smoke,
    },
)
upload_receipts.append(
    persist_evidence(training_bundle, phase="verified training artifacts", first_upload=True)
)

## 9. Exercise the exact adapter through FastAPI and Pydantic once

This intentionally uses the predeclared synthetic validation case `MF-VA-0001`. The helper revalidates the complete validation dataset manifest, split hash, case-set hash, training manifest, and adapter hashes before making one real cache-only adapter request through FastAPI's in-process `TestClient`. This is serving-contract evidence only, not model-quality, improvement, or calibration evidence.

In [ ]:
adapter_api_smoke = audit_root / "adapter-api-smoke.json"
subprocess.run(
    [
        str(VENV_PYTHON),
        str(HELPER_PATH),
        "adapter-api-smoke",
        "--training-manifest",
        str(manifest_path),
        "--dataset-root",
        str(PROJECT_ROOT / "data/iam_triage_v1"),
        "--output",
        str(adapter_api_smoke),
        "--project-root",
        str(PROJECT_ROOT),
        "--source-commit",
        SOURCE_COMMIT,
        "--run-id",
        RUN_ID,
        "--timeout-seconds",
        "120",
    ],
    cwd=PROJECT_ROOT,
    env=run_env,
    check=True,
)
api_smoke_artifact = json.loads(adapter_api_smoke.read_text(encoding="utf-8"))
if api_smoke_artifact.get("response_status_code") != 200:
    raise RuntimeError("Adapter-backed FastAPI smoke did not return HTTP 200")
if api_smoke_artifact.get("x_modelforge_model") != "small-iam-triage-v1":
    raise RuntimeError("Adapter-backed FastAPI smoke has the wrong model header")
if api_smoke_artifact.get("validation_case_id") != "MF-VA-0001":
    raise RuntimeError("Adapter-backed FastAPI smoke used an unexpected case")
final_bundle = prepare_evidence_bundle(
    manifest_path,
    BUNDLE_PARENT / "final-evidence",
    source_commit=SOURCE_COMMIT,
    run_id=RUN_ID,
    environment_files=environment_files,
    evaluation_files={"base-preflight.json": preflight_result},
    audit_files={
        "response-only-mask-audit.json": response_mask_audit,
        "confidence-routing-smoke.json": routing_smoke,
        "adapter-api-smoke.json": adapter_api_smoke,
    },
)
upload_receipts.append(
    persist_evidence(
        final_bundle,
        phase="minimal training, preflight, tokenizer, FastAPI, and routing evidence",
        first_upload=False,
    )
)
print("Verified one predeclared adapter-backed FastAPI/Pydantic request")

## 10. Download the exact private repository commit and re-hash every file

In [ ]:
def verify_private_upload() -> tuple[str, int]:
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("HF_TOKEN is unavailable for final verification")
    api = HfApi(token=token)
    try:
        current_info = api.repo_info(repo_id=ARTIFACT_REPO, repo_type="dataset")
        if not bool(getattr(current_info, "private", False)):
            raise RuntimeError("Artifact repository is no longer private")
        if getattr(current_info, "sha", None) != repository_head:
            raise RuntimeError("Artifact repository changed before final verification")
        verification_container = Path("/content/modelforge-remote-verification")
        if verification_container.exists() or verification_container.is_symlink():
            raise RuntimeError("Remote verification directory already exists")
        snapshot_root = Path(
            snapshot_download(
                repo_id=ARTIFACT_REPO,
                repo_type="dataset",
                revision=repository_head,
                local_dir=str(verification_container),
                allow_patterns=[f"{REMOTE_RUN_PATH}/**"],
                token=token,
            )
        )
        downloaded_bundle = snapshot_root / REMOTE_RUN_PATH
        local_hashes = verify_downloaded_bundle(final_bundle)
        remote_hashes = verify_downloaded_bundle(downloaded_bundle)
        if remote_hashes != local_hashes:
            raise RuntimeError("Commit-pinned remote bundle hashes differ from local evidence")
        if sha256_file(downloaded_bundle / "evidence-index.sha256") != sha256_file(
            final_bundle / "evidence-index.sha256"
        ):
            raise RuntimeError("Remote evidence index differs from the local index")
        url = (
            f"https://huggingface.co/datasets/{ARTIFACT_REPO}/tree/"
            f"{repository_head}/{REMOTE_RUN_PATH}"
        )
        return url, len(remote_hashes)
    finally:
        token = ""
        api = None


final_url, verified_file_count = verify_private_upload()
print(f"Verified {verified_file_count} evidence files at the exact private commit")
print(final_url)

## 11. Stop before locked-test evaluation

A successful final cell proves that the one-epoch adapter, manifest, one-case base preflight, environment record, full 500-row tokenizer-mask audit, predeclared-case adapter-backed FastAPI/Pydantic smoke, and explicitly uncalibrated routing control-flow smoke match the exact files in the private Hugging Face commit. This deliberately minimal run establishes implementation evidence only; it does not establish model-quality improvement, calibrated routing, benchmark performance, or production readiness. No full validation generation or locked-test evaluation is performed.

When finished, select **Runtime → Disconnect and delete runtime**. That removes the temporary Qwen base-model cache. Revoke the short-lived Hugging Face token after confirming the private evidence repository.